In [1]:
import numpy as np
import seaborn as sns
import tarfile
import pandas as pd
import concurrent.futures
import librosa
from tqdm import tqdm
import h5py
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

Ten skrypt służy dla tworzenia danych do treningu i ewaluacji modelu CNN UBM analizującego mel-spektrogramy.

Wybór 100 mówców dla danych treningowych oraz 50 dla ewalucji.

In [2]:
df_expanded = pd.read_csv('/kaggle/input/df-exp/df_expanded.csv', sep=',')
df_expanded = df_expanded.drop(columns=[ 'SUBSET', 'Pitch Tracking'])
df_expanded['SEX'] = df_expanded['SEX'].str.strip()

females50 = df_expanded[df_expanded['SEX'] == 'F']['NAME'].value_counts().iloc[0:50].index #top 50 mówców 
males50 = df_expanded[df_expanded['SEX'] == 'M']['NAME'].value_counts().iloc[0:50].index
females25 = df_expanded[df_expanded['SEX'] == 'F']['NAME'].value_counts().iloc[50:75].index # top 50-75 mówców
males25 = df_expanded[df_expanded['SEX'] == 'M']['NAME'].value_counts().iloc[50:75].index

classes50 = females25.append(males25) 

classes100 = females50.append(males50) 

df_filtered = df_expanded[df_expanded['NAME'].isin(classes100.append(classes50))].copy()

Cięcie nagrań wybranych mówców na okienka 1-sekundowe.

In [3]:
sr = 8000
window_size_sec = 1
samples_per_window = window_size_sec * sr
df_final = pd.DataFrame()
window_data = []

for speaker in tqdm(classes100.append(classes50), desc="Processing speakers", total=len(classes100.append(classes50))): #pasek śledzenia czasu
    speaker_data = df_filtered[df_filtered['NAME'] == speaker]
    speaker_id = speaker_data['ID'].iloc[0]
    
    full_audio = np.concatenate([
        librosa.load(file, sr=sr)[0] for file in speaker_data['FLAC File'] #łączenie wszystkich nagrań mówcy
    ])
    
    num_windows = len(full_audio) // samples_per_window #ilość okienek
    for i in range(num_windows):
        start_idx = i * samples_per_window 
        end_idx = start_idx + samples_per_window
        window = full_audio[start_idx:end_idx]
        
        if len(window) == samples_per_window:
            window_data.append({
                'ID': speaker_id,
                'NAME': speaker,
                'audio_window': window,
            })

df_final = pd.DataFrame(window_data)

Processing speakers: 100%|██████████| 150/150 [06:05<00:00,  2.43s/it]


Wyliczanie MFCC dla poszczególnych okienek oraz pochodne (delta).

In [4]:
n_mfcc = 13
n_mels = 26

def process_file_with_deltas(row):
    y = row['audio_window']
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_mels=n_mels)  # Obliczanie MFCC
    mfcc_delta = librosa.feature.delta(mfcc, order=1)                      # Pierwsza pochodna
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)                     # Druga pochodna
    mfcc_combined = np.hstack((mfcc.T, mfcc_delta.T, mfcc_delta2.T))       # Połączenie macierzy
    return mfcc.T, mfcc_combined

mfcc_list = []
mfcc_delta_combined_list = []

for _, row in tqdm(df_final.iterrows(), total=len(df_final)):  # Pasek śledzenia
    mfcc, mfcc_combined = process_file_with_deltas(row)
    mfcc_list.append(mfcc)
    mfcc_delta_combined_list.append(mfcc_combined)

df_final['MFCC'] = mfcc_list
df_final['MFCC_Delta'] = mfcc_delta_combined_list

df_final.to_csv('final.csv', index=False)
df_final.head()

100%|██████████| 224641/224641 [25:57<00:00, 144.22it/s]


,ID,NAME,audio_window,MFCC,MFCC_Delta
0,211,shanda_w,"[0.016172696, 0.024148379, 0.01841107, 0.03984...","[[-131.5245, 0.32613522, 12.214758, 14.360971,...","[[-131.5245, 0.32613522, 12.214758, 14.360971,..."
1,211,shanda_w,"[0.027316898, 0.037306715, 0.032587398, 0.0312...","[[-68.74807, 17.001019, 26.302664, 16.494469, ...","[[-68.74807, 17.001019, 26.302664, 16.494469, ..."
2,211,shanda_w,"[0.023928262, 0.021538164, 0.030577034, 0.0182...","[[-155.19469, 19.945545, 10.978376, 13.199591,...","[[-155.19469, 19.945545, 10.978376, 13.199591,..."
3,211,shanda_w,"[0.043404847, 0.038200647, 0.039111435, 0.0419...","[[-44.77331, 28.831783, -5.331954, 0.014904141...","[[-44.77331, 28.831783, -5.331954, 0.014904141..."
4,211,shanda_w,"[0.024801528, 0.024052162, 0.024652848, 0.0226...","[[-102.93453, -13.687842, 21.460638, -2.764333...","[[-102.93453, -13.687842, 21.460638, -2.764333..."


**Tworzenie zbioru treningowego**

In [8]:
df_train = df_final[df_final['NAME'].isin(classes100)].copy()
label_encoder = LabelEncoder()


X = df_train['MFCC']
X_delta = df_train['MFCC_Delta']
y = df_train.drop(columns=['MFCC', 'audio_window', 'ID', 'MFCC_Delta'])
y = label_encoder.fit_transform(y.values.ravel())
y = to_categorical(y) #kodowanie zmiennej nominalnej


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
X_train_delta, X_val_delta = train_test_split(X_delta, test_size=0.25, random_state=42)


X_train = np.array([MFCC.reshape(X_train.iloc[0].shape[0], X_train.iloc[0].shape[1]) for MFCC in X_train]) #przekształcenie danych do postaci interpretowalnej przez model
X_val = np.array([MFCC.reshape(X_val.iloc[0].shape[0], X_val.iloc[0].shape[1]) for MFCC in X_val])

X_train_delta = np.array([MFCC.reshape(X_train_delta.iloc[0].shape[0], X_train_delta.iloc[0].shape[1]) for MFCC in X_train_delta])
X_val_delta = np.array([MFCC.reshape(X_val_delta.iloc[0].shape[0], X_val_delta.iloc[0].shape[1]) for MFCC in X_val_delta])

**Tworzenie zbioru testowego**

In [9]:
df_emb = df_final[df_final['NAME'].isin(classes50)].copy()
df_emb['NAME'] = label_encoder.fit_transform(df_emb['NAME'])
classes_comparison = df_emb['NAME'].unique()

Tworzenie kolumny "label", dla póżniejszego łączenia okienek.

In [10]:
window_size_train_sec = 40  
window_size_test_sec = 10  

df_emb['label'] = ""

for speaker in tqdm(classes_comparison, desc="Processing speakers", total=len(classes_comparison)):
    speaker_df = df_emb[df_emb['NAME'] == speaker].copy()
    speaker_id = df_emb[df_emb['NAME'] == speaker]['ID'].iloc[0]  

    for idx in range(window_size_train_sec):
        if idx < len(speaker_df):
            df_emb.loc[speaker_df.index[idx], 'label'] = f"enrollment_{speaker_id}"   #40 okienek z lablem enrollment

    remaining_segments = len(speaker_df) - window_size_train_sec
    num_test_groups = remaining_segments // window_size_test_sec

    for i in range(num_test_groups):
        for n in range(window_size_test_sec):
            idx = window_size_train_sec + i * window_size_test_sec + n
            if idx < len(speaker_df):
                df_emb.loc[speaker_df.index[idx], 'label'] = f"test{i+1}_{speaker_id}"  #każde 10 następnych okienek mają label test{n}

df_emb = df_emb[df_emb['label'] != ""]

Processing speakers: 100%|██████████| 50/50 [00:15<00:00,  3.19it/s]


Wyciąganie z ramki danych kluczowych informacji.

In [13]:
labels = np.array(df_emb['label'])
ID = np.array(df_emb['ID'])
X_emb = df_emb['MFCC']
X_emb_delta = df_emb['MFCC_Delta']

X_emb = np.array([MFCC.reshape(X_emb.iloc[0].shape[0], X_emb.iloc[0].shape[1]) for MFCC in X_emb])
X_emb_delta = np.array([MFCC.reshape(X_emb_delta.iloc[0].shape[0], X_emb_delta.iloc[0].shape[1]) for MFCC in X_emb_delta])

**Zapisywanie zbiorów**

In [14]:
with h5py.File('data.h5', 'w') as f:
    f.create_dataset('train_classes', data=np.asarray(classes100))
    f.create_dataset('emb_classes', data=np.asarray(classes50))
    f.create_dataset('X_train', data=X_train)
    f.create_dataset('X_train_delta', data=X_train_delta)
    f.create_dataset('y_train', data=y_train)
    f.create_dataset('X_val_delta', data=X_val_delta)
    f.create_dataset('X_val', data=X_val)
    f.create_dataset('y_val', data=y_val)
    f.create_dataset('X_emb', data=X_emb)
    f.create_dataset('X_emb_delta', data=X_emb_delta)
    f.create_dataset('labels', data=labels)
    f.create_dataset('ID', data=ID)